# Playground de cutouts de Legacy Survey

## 0. Objetivo de este playground

Este notebook **no forma parte del pipeline de producción** (`src/config.py`, `src/data/legacy_cutouts.py`,
`src/data/download_cutouts.py`). Es un laboratorio experimental, pequeño y reproducible, para decidir
**cómo** deben descargarse las ~10,000 imágenes MaNGA antes de lanzar la descarga masiva.

Tres preguntas guían el experimento:

1. **¿Qué tan lejos debemos encuadrar cada galaxia?** (campo de visión / `fov_factor`)
2. **¿Qué release/capa conviene usar: DR9 o DR11?**
3. **¿Conviene el render de color por defecto o una composición `grz` explícita?**

Regla general: **cambiar una sola variable por experimento**. Por eso el notebook resuelve primero el
zoom (Experimento A), después el release (Experimento B) y finalmente el color (Experimento C), siempre
sobre la **misma muestra de galaxias**.

Todo lo que se descarga aquí se guarda en `data/legacy_cutouts_playground/`, una carpeta independiente
del dataset oficial (`data/legacy_cutouts/`), que **no se toca**. Cuando exista una configuración
ganadora, esa decisión (y solo esa) se migrará a `src/config.py` / `src/data/legacy_cutouts.py`.

Este notebook implementa su propio downloader mínimo (no reutiliza `legacy_cutouts.py` para no acoplar
el playground al pipeline oficial), pero sí reutiliza constantes y funciones de carga/etiquetado ya
validadas en `src/`.

## 1. Imports y configuración experimental

Todas las variables que se pueden modificar en este playground viven juntas en la siguiente celda,
para que quede claro qué es "configuración del experimento" y qué es código de soporte.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Repo en el sys.path para poder reutilizar src.config / src.data.morfologia,
# sin depender de dónde se lance jupyter (ver README.md, sección Convenciones).
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MORFOLOGIA_CSV
from src.data.morfologia import derive_ttype_label, load_morfologia

# Funciones de descarga/visualización reutilizables (ver notebook/julio/playground_utils.py).
NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "playground_utils.py").exists() else PROJECT_ROOT / "notebook" / "julio"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from playground_utils import (
    build_cutout_url,
    compute_pixscale,
    describe_image_quality,
    download_batch,
    make_review_table,
    plot_comparison_grid,
    prepare_jobs,
)

# --- Reproducibilidad ---
RANDOM_SEED = 42

# --- Muestra experimental (sección 3) ---
SAMPLE_SIZE = 30  # ~24-36 galaxias: suficiente para inspección visual, no para estadística

# --- Resolución fija durante todo el playground (sección 5) ---
# No exploramos 128/224/512 px aquí: es otra variable, y este playground
# solo estudia qué contiene el encuadre, no cuántos píxeles tiene.
SIZE_PX = 256

# --- Experimento A: campo de visión (sección 6) ---
FOV_FACTORS = [4, 5, 6, 8]
GALAXIES_PER_GRID = 6  # paginación de las cuadrículas comparativas

# Límites de zoom razonables, equivalentes a los de src/config.py
# (CUTOUT_PIXSCALE_MIN/MAX), para no pedir ampliaciones ni encuadres absurdos.
PIXSCALE_MIN = 0.1
PIXSCALE_MAX = 0.6

# Capa/composición fija usada en el Experimento A (una sola variable a la vez):
# DR11 con banda grz explícita, para no mezclar zoom con release/color.
BASE_LAYER = "ls-dr11"
BASE_BANDS = "grz"

# --- Experimento B: DR9 vs DR11 (sección 9) ---
# FOV provisional elegido a partir de la inspección visual del Experimento A.
# Editar este valor manualmente después de revisar las cuadrículas de la sección 6.
PROVISIONAL_FOV_FACTOR = 6
RELEASES = {"DR9": "ls-dr9", "DR11": "ls-dr11"}

# --- Experimento C: composición de color (sección 10) ---
# None => no se envía el parámetro "bands" (render por defecto del Viewer).
COLOR_MODES = {"grz": "grz", "default": None, "g": "g", "r": "r", "z": "z"}
# Release/FOV fijos elegidos provisionalmente tras los experimentos A y B.
PROVISIONAL_LAYER = "ls-dr11"

# El servicio empezó a responder 429 (Too Many Requests) con 6 hilos y sin
# espaciar peticiones: bajamos la concurrencia y forzamos un ritmo mínimo
# entre requests (MIN_REQUEST_INTERVAL_S), compartido por todos los hilos.
MAX_CONCURRENT_REQUESTS = 3  # servicio público: ser moderados
MIN_REQUEST_INTERVAL_S = 1.0  # espacio mínimo entre requests, sin importar el hilo
TIMEOUT_S = 60
MAX_RETRIES = 3
RETRY_BACKOFF_S = 2.0
RETRY_BACKOFF_429_S = 10.0  # espera base tras un 429, mucho mayor que un error normal
OVERWRITE = False  # no volver a descargar archivos cacheados

# --- Carpetas del playground (independientes de data/legacy_cutouts/) ---
PLAYGROUND_DIR = PROJECT_ROOT / "data" / "legacy_cutouts_playground"
PLAYGROUND_INDEX_CSV = PLAYGROUND_DIR / "playground_index.csv"

print(f"PROJECT_ROOT     = {PROJECT_ROOT}")
print(f"MORFOLOGIA_CSV   = {MORFOLOGIA_CSV}")
print(f"PLAYGROUND_DIR   = {PLAYGROUND_DIR}")

## 2. Carga de `morfologia.csv`

Reutilizamos `load_morfologia()` y `derive_ttype_label()` de `src/data/morfologia.py` (ya validadas en
`notebook/exploration_morfologia.ipynb`), y aplicamos los filtros que pide este playground:

- descartar `fuera_de_rango` (Ttype fuera de las reglas temprana/tardía);
- descartar galaxias sin `PETRO_TH90` (sin ese radio no podemos calcular el zoom por galaxia);
- descartar filas sin coordenadas (`objra`/`objdec`) válidas.

In [ ]:
raw = load_morfologia(MORFOLOGIA_CSV)
labeled = derive_ttype_label(raw)

pool = labeled[labeled["label"].isin(["temprana", "tardia"])].copy()
pool = pool.dropna(subset=["PETRO_TH90"])
pool = pool.dropna(subset=["objra", "objdec"])
pool = pool[pool["objra"].between(0, 360) & pool["objdec"].between(-90, 90)]

print(f"Filas totales en morfologia.csv : {len(labeled)}")
print(f"Filas utilizables tras filtros   : {len(pool)}")
pool["label"].value_counts()

## 3. Comprender `PETRO_TH90`

`PETRO_TH90` es el **radio de Petrosian al 90%**: el radio angular (en segundos de arco) dentro del cual
cae el 90% del flujo luminoso de la galaxia. Es, en la práctica, una medida del **tamaño angular
aparente** de la galaxia vista desde la Tierra.

¿Por qué nos importa para descargar imágenes? Las galaxias de la muestra MaNGA tienen tamaños angulares
muy distintos entre sí. Si pedimos el mismo campo de visión (mismo zoom) para todas:

- las galaxias grandes quedarán cortadas (brazos, disco o halo fuera del cuadro);
- las galaxias pequeñas quedarán como un puñado de píxeles perdidos en fondo de cielo.

Por eso el campo de visión (FOV) de cada cutout se calcula **en función de `PETRO_TH90`**, en vez de
usar un zoom angular fijo (sección 4). Antes de decidir el `fov_factor` conviene ver cuánto varía
`PETRO_TH90` en la muestra.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(pool["PETRO_TH90"], bins=40, color="#4C72B0")
axes[0].set_xlabel("PETRO_TH90 (arcsec)")
axes[0].set_ylabel("N galaxias")
axes[0].set_title("Distribución de PETRO_TH90")

pool.boxplot(column="PETRO_TH90", by="label", ax=axes[1])
axes[1].set_xlabel("label")
axes[1].set_ylabel("PETRO_TH90 (arcsec)")
axes[1].set_title("PETRO_TH90 por clase")
plt.suptitle("")
plt.tight_layout()
plt.show()

quantiles = pool["PETRO_TH90"].quantile([0.0, 0.25, 0.5, 0.75, 1.0])
print("Cuantiles de PETRO_TH90 (arcsec):")
print(quantiles)

print("\n5 galaxias más pequeñas:")
display(pool.nsmallest(5, "PETRO_TH90")[["name", "label", "Ttype", "PETRO_TH90"]])

print("5 galaxias más grandes:")
display(pool.nlargest(5, "PETRO_TH90")[["name", "label", "Ttype", "PETRO_TH90"]])

## 4. Selección de la muestra fija

La muestra debe cubrir **deliberadamente** galaxias pequeñas, medianas y grandes según `PETRO_TH90` (no
solo temprana/tardía), porque justamente queremos comprobar si una misma regla de encuadre funciona para
tamaños muy distintos. Usamos **cuantiles** (tercios) de `PETRO_TH90` en vez de límites arbitrarios, y
armamos una cuadrícula 2 (label) × 3 (tamaño) con ~5 galaxias por celda (~30 en total).

La semilla (`RANDOM_SEED = 42`) hace la muestra reproducible: volver a correr esta celda siempre
selecciona las mismas galaxias.

In [ ]:
pool = pool.copy()
pool["size_group"] = pd.qcut(
    pool["PETRO_TH90"], q=3, labels=["pequeña", "mediana", "grande"]
)

n_cells = 2 * 3  # 2 labels x 3 tamaños
per_cell = max(1, SAMPLE_SIZE // n_cells)

parts = []
for (label, size_group), group in pool.groupby(["label", "size_group"], observed=True):
    parts.append(group.sample(n=min(per_cell, len(group)), random_state=RANDOM_SEED))

sample = pd.concat(parts).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Tamaño final de la muestra: {len(sample)}")
print("\nDistribución label x size_group:")
display(pd.crosstab(sample["label"], sample["size_group"]))

sample[["name", "label", "Ttype", "PETRO_TH90", "objra", "objdec", "size_group"]]

# Selección manual de muestras específicas
sample_selected = ["manga-8147-3704","manga-8311-6103","manga-8620-12704","manga-8937-9101",
                "manga-8549-12701","manga-8486-12701","manga-8547-12703","manga-8561-3704", "manga-7962-6102", "manga-7992-9101"]

sample_selected = pool[pool.name.isin(sample_selected)].copy().reset_index(drop=True)

## 5. Downloader mínimo

### FOV y pixscale (por qué no usamos un zoom fijo)

- **FOV** (field of view): el campo de visión total del cutout, en segundos de arco de lado.
- **pixscale**: cuántos segundos de arco de cielo representa **un solo píxel** de la imagen.

El servicio de legacysurvey.org no recibe un "zoom" directamente: recibe `size` (lado en píxeles, fijo
en `SIZE_PX = 256`) y `pixscale`. La relación es:

$$\text{FOV} = \text{fov\_factor} \times \text{PETRO\_TH90}$$
$$\text{pixscale} = \frac{\text{FOV}}{\text{SIZE\_PX}}$$

Con `fov_factor` fijo, una galaxia con `PETRO_TH90` grande pide más arcsec por imagen (FOV mayor) y por
lo tanto más arcsec por píxel (pixscale mayor): la imagen "se aleja" lo justo para que quepa completa.
Una galaxia con `PETRO_TH90` pequeño pide un pixscale menor: la imagen "se acerca" para no desperdiciar
píxeles en fondo vacío. Usar el mismo `pixscale` para todas las galaxias produciría exactamente los dos
problemas que queremos evitar: galaxias grandes cortadas y galaxias pequeñas reducidas a unos pocos
píxeles.

`pixscale` se acota a `[PIXSCALE_MIN, PIXSCALE_MAX]` para no pedir ampliaciones absurdas ni encuadres
tan abiertos que la galaxia desaparezca en el fondo.

**Ejemplo numérico:** una galaxia con `PETRO_TH90 = 8` arcsec y `fov_factor = 4` pide
`FOV = 4 × 8 = 32` arcsec de lado, y con `SIZE_PX = 256` eso da `pixscale = 32 / 256 = 0.125` arcsec/px.

In [ ]:
def compute_pixscale(petro_th90, fov_factor, size_px=SIZE_PX):
    """Calcula pixscale (arcsec/px) a partir de PETRO_TH90 y el fov_factor.

    Ver explicación en la celda markdown anterior. Acota el resultado a
    [PIXSCALE_MIN, PIXSCALE_MAX] para no pedir zooms absurdos.
    """
    fov_arcsec = fov_factor * petro_th90
    pixscale = fov_arcsec / size_px
    return float(np.clip(pixscale, PIXSCALE_MIN, PIXSCALE_MAX))


def build_cutout_url(ra, dec, pixscale, size_px=SIZE_PX, layer=BASE_LAYER, bands=None):
    """Construye la URL del jpeg-cutout service de legacysurvey.org.

    `bands=None` omite el parámetro `bands`: el Viewer usa su render de
    color por defecto (relevante para el Experimento C, sección 10).
    """
    url = (
        f"{LEGACY_CUTOUT_BASE_URL}/jpeg-cutout"
        f"?ra={ra}&dec={dec}&size={size_px}&layer={layer}&pixscale={pixscale:.4f}"
    )
    if bands:
        url += f"&bands={bands}"
    return url


# Ejemplo manual antes de la descarga masiva (sección 11 de la especificación).
_example = sample.iloc[0]
_example_pixscale = compute_pixscale(_example["PETRO_TH90"], fov_factor=4)
_example_url = build_cutout_url(
    _example["objra"], _example["objdec"], _example_pixscale, layer=BASE_LAYER, bands=BASE_BANDS
)
print(f"Galaxia de ejemplo : {_example['name']} (PETRO_TH90={_example['PETRO_TH90']:.2f} arcsec)")
print(f"pixscale calculado : {_example_pixscale:.4f} arcsec/px")
print(f"URL construida     : {_example_url}")

### Descarga concurrente

> Mientras una petición espera respuesta del servidor, podemos iniciar otras, sin necesidad de
> descargar las imágenes una por una. Eso es **concurrencia de red**, no paralelismo de CPU: no estamos
> usando más núcleos, solo aprovechando el tiempo de espera de I/O.

`aiohttp`/`httpx` (la opción sugerida para `asyncio`) no están en `requirements.txt` de este proyecto.
Para no añadir una dependencia nueva solo para el playground, implementamos la concurrencia con
`concurrent.futures.ThreadPoolExecutor` y `requests` (ya es dependencia del proyecto): cada hilo espera
su propia respuesta HTTP, y `MAX_CONCURRENT_REQUESTS` limita cuántas peticiones están en vuelo a la vez,
exactamente el mismo rol que cumpliría un `asyncio.Semaphore`. Esto es, en la práctica, el "fallback"
que pide la especificación si el entorno presenta problemas con `asyncio` en el notebook.

Cada descarga:

- se salta si el archivo ya existe en disco (`OVERWRITE=False`) → estado `"cached"`;
- reintenta hasta `MAX_RETRIES` veces con backoff lineal si falla la red o el HTTP;
- valida que el archivo se pueda abrir como imagen antes de marcarlo `"downloaded"`;
- nunca cancela el resto del lote: los errores quedan registrados por galaxia.

`MAX_CONCURRENT_REQUESTS` acota **cuántas peticiones están en vuelo a la vez**, pero no evita que varios
hilos disparen su petición casi al mismo instante. Si el servicio responde `429 Too Many Requests`, es
señal de que hace falta espaciar también **el ritmo** de peticiones, no solo limitar la concurrencia. Por
eso añadimos un `RateLimiter` compartido (`MIN_REQUEST_INTERVAL_S`) que fuerza un espacio mínimo entre
requests sin importar qué hilo la haga, y un backoff mucho más largo específico para 429
(`RETRY_BACKOFF_429_S`, respetando el header `Retry-After` si el servidor lo envía).

In [ ]:
def playground_path(experiment, subfolder, name):
    """Ruta destino dentro de data/legacy_cutouts_playground/, por experimento."""
    return PLAYGROUND_DIR / experiment / subfolder / f"{name}.jpg"


def prepare_jobs(sample_df, experiment, subfolder, layer, bands, fov_factor, size_px=SIZE_PX):
    """Arma la lista de "jobs" (metadatos + URL + ruta local) para un lote de descarga.

    Una fila por galaxia. `experiment`/`subfolder` determinan dónde se guarda el
    archivo dentro de data/legacy_cutouts_playground/ (sección 13/14).
    """
    jobs = []
    for row in sample_df.to_dict("records"):
        pixscale = compute_pixscale(row["PETRO_TH90"], fov_factor=fov_factor, size_px=size_px)
        url = build_cutout_url(
            row["objra"], row["objdec"], pixscale, size_px=size_px, layer=layer, bands=bands
        )
        local_path = playground_path(experiment, subfolder, row["name"])
        jobs.append(
            {
                "name": row["name"],
                "label": row["label"],
                "Ttype": row["Ttype"],
                "objra": row["objra"],
                "objdec": row["objdec"],
                "PETRO_TH90": row["PETRO_TH90"],
                "size_group": row.get("size_group"),
                "experiment": experiment,
                "layer": layer,
                "bands_or_color_mode": bands if bands else "default",
                "fov_factor": fov_factor,
                "fov_arcsec": round(pixscale * size_px, 2),
                "pixscale": round(pixscale, 4),
                "size_px": size_px,
                "url": url,
                "local_path": str(local_path),
            }
        )
    return jobs


class RateLimiter:
    """Fuerza un espacio mínimo entre requests, compartido por todos los hilos.

    `MAX_CONCURRENT_REQUESTS` acota cuántas peticiones están en vuelo a la vez,
    pero no cuántas por segundo. El servicio empezó a responder 429 (Too Many
    Requests) porque varios hilos disparaban peticiones casi simultáneas; este
    limitador espacia cada `requests.get` al menos `min_interval_s`.
    """

    def __init__(self, min_interval_s):
        self.min_interval_s = min_interval_s
        self._lock = threading.Lock()
        self._last_request_at = 0.0

    def wait(self):
        with self._lock:
            now = time.time()
            wait_s = self._last_request_at + self.min_interval_s - now
            if wait_s > 0:
                time.sleep(wait_s)
            self._last_request_at = time.time()


_rate_limiter = RateLimiter(MIN_REQUEST_INTERVAL_S)


def download_one_cutout(
    job,
    timeout_s=TIMEOUT_S,
    max_retries=MAX_RETRIES,
    backoff_s=RETRY_BACKOFF_S,
    backoff_429_s=RETRY_BACKOFF_429_S,
):
    """Descarga (o recupera de caché) un único cutout y valida la imagen.

    Devuelve una fila de índice completa (sección 14), nunca lanza excepción:
    los errores quedan en `status="error"` para no interrumpir el resto del lote.
    """
    dest = Path(job["local_path"])
    record = dict(job)

    if dest.exists() and not OVERWRITE:
        record.update(status="cached", http_status=None, bytes=dest.stat().st_size, error=None, download_time_s=0.0)
        return record

    dest.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    last_error = None
    http_status = None

    for attempt in range(1, max_retries + 1):
        _rate_limiter.wait()
        try:
            response = requests.get(job["url"], timeout=timeout_s)
            http_status = response.status_code
            if http_status == 429:
                # Respetar Retry-After si el servidor lo manda; si no, backoff largo.
                retry_after = response.headers.get("Retry-After")
                wait_s = float(retry_after) if retry_after else backoff_429_s * attempt
                raise requests.HTTPError(f"429 Too Many Requests (retry en {wait_s:.1f}s)", response=response)
            response.raise_for_status()
            content = response.content
            if not content:
                raise ValueError("respuesta vacía")
            dest.write_bytes(content)
            with Image.open(dest) as im:
                im.verify()  # valida que sea una imagen legible
            record.update(
                status="downloaded",
                http_status=http_status,
                bytes=len(content),
                error=None,
                download_time_s=round(time.time() - t0, 3),
            )
            return record
        except Exception as error:  # red, HTTP, timeout o imagen corrupta
            last_error = error
            if attempt < max_retries:
                if http_status == 429:
                    retry_after = getattr(getattr(error, "response", None), "headers", {}).get("Retry-After")
                    time.sleep(float(retry_after) if retry_after else backoff_429_s * attempt)
                else:
                    time.sleep(backoff_s * attempt)

    record.update(
        status="error",
        http_status=http_status,
        bytes=None,
        error=str(last_error),
        download_time_s=round(time.time() - t0, 3),
    )
    return record


def download_batch(jobs, max_concurrent=MAX_CONCURRENT_REQUESTS, verbose=True):
    """Descarga un lote de jobs con concurrencia acotada (ver celda anterior)."""
    records = []
    with ThreadPoolExecutor(max_workers=max_concurrent) as executor:
        futures = {executor.submit(download_one_cutout, job): job for job in jobs}
        for i, future in enumerate(as_completed(futures), start=1):
            record = future.result()
            records.append(record)
            if verbose and record["status"] == "error":
                print(f"  ! {record['name']} ({record['experiment']}): {record['error']}")
    index = pd.DataFrame.from_records(records)
    if verbose:
        print(
            f"Lote completo: {len(index)} jobs -> "
            f"{(index['status'] == 'downloaded').sum()} descargadas, "
            f"{(index['status'] == 'cached').sum()} cacheadas, "
            f"{(index['status'] == 'error').sum()} con error"
        )
    return index

In [ ]:
def plot_comparison_grid(index_df, col_field, col_values, col_titles=None,
                          galaxies_per_grid=GALAXIES_PER_GRID, suptitle=""):
    """Cuadrícula galaxia (fila) x configuración (columna), paginada.

    `index_df` debe traer una fila por (galaxia, valor de `col_field`), con
    columnas name/label/Ttype/PETRO_TH90/local_path/status. Reutilizada por
    los tres experimentos (A: fov_factor, B: layer, C: bands_or_color_mode).
    """
    col_titles = col_titles or [str(v) for v in col_values]
    names = index_df["name"].drop_duplicates().tolist()
    pages = [names[i : i + galaxies_per_grid] for i in range(0, len(names), galaxies_per_grid)]

    for page_i, page_names in enumerate(pages, start=1):
        n_rows, n_cols = len(page_names), len(col_values)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3.2 * n_rows), squeeze=False)

        for r, name in enumerate(page_names):
            galaxy_rows = index_df[index_df["name"] == name]
            meta = galaxy_rows.iloc[0]
            for c, col_value in enumerate(col_values):
                ax = axes[r][c]
                match = galaxy_rows[galaxy_rows[col_field] == col_value]
                if len(match) and match.iloc[0]["status"] != "error" and Path(match.iloc[0]["local_path"]).exists():
                    ax.imshow(Image.open(match.iloc[0]["local_path"]))
                else:
                    ax.text(0.5, 0.5, "sin imagen", ha="center", va="center", fontsize=9)
                ax.set_xticks([])
                ax.set_yticks([])
                if r == 0:
                    ax.set_title(str(col_titles[c]), fontsize=11)
                if c == 0:
                    label_txt = (
                        f"{meta['name']}\n{meta['label']} (T={meta['Ttype']})\n"
                        f"PETRO_TH90={meta['PETRO_TH90']:.1f}\""
                    )
                    ax.set_ylabel(label_txt, fontsize=8, rotation=0, ha="right", va="center")

        fig.suptitle(f"{suptitle} (página {page_i}/{len(pages)})", fontsize=13)
        plt.tight_layout()
        plt.show()


def make_review_table(index_df, extra_group_field, extra_group_values):
    """DataFrame vacío de revisión manual (sección 8): review_score y notes en blanco."""
    rows = []
    for name, group in index_df.groupby("name"):
        meta = group.iloc[0]
        for value in extra_group_values:
            rows.append(
                {
                    "name": name,
                    "label": meta["label"],
                    "size_group": meta["size_group"],
                    extra_group_field: value,
                    "review_score": pd.NA,  # -2 .. +2, ver escala en la celda markdown
                    "notes": "",
                }
            )
    return pd.DataFrame(rows)

## 6. Experimento A — campo de visión / zoom

Este es el experimento **más importante**. Mantenemos constantes la muestra, `SIZE_PX`, el release
(`BASE_LAYER = "ls-dr11"`) y la composición de color (`BASE_BANDS = "grz"`), y variamos únicamente
`fov_factor` en `FOV_FACTORS = [4, 5, 6, 8]`.

Buscamos el `fov_factor` que evite los dos extremos:

- **demasiado cerca**: brazos cortados, halo/disco fuera del cuadro, galaxia tocando los bordes;
- **demasiado lejos**: la galaxia ocupa pocos píxeles, estructura interna difícil de distinguir.

In [ ]:
jobs_fov = []
for fov_factor in FOV_FACTORS:
    jobs_fov += prepare_jobs(
        sample,
        experiment="fov",
        subfolder=f"fov{fov_factor}",
        layer=BASE_LAYER,
        bands=BASE_BANDS,
        fov_factor=fov_factor,
    )

index_fov = download_batch(jobs_fov)
index_fov.head()

In [ ]:
plot_comparison_grid(
    index_fov,
    col_field="fov_factor",
    col_values=FOV_FACTORS,
    col_titles=[f"FOV={f}" for f in FOV_FACTORS],
    suptitle="Experimento A: fov_factor",
)

In [ ]:
# Revisar conjunto específico de muestras

jobs_fov = []
for fov_factor in FOV_FACTORS:
    jobs_fov += prepare_jobs(
        sample_selected,
        experiment="fov",
        subfolder=f"fov{fov_factor}",
        layer=BASE_LAYER,
        bands=BASE_BANDS,
        fov_factor=fov_factor,
    )

index_fov = download_batch(jobs_fov)
index_fov.head()

In [ ]:
plot_comparison_grid(
    index_fov,
    col_field="fov_factor",
    col_values=FOV_FACTORS,
    col_titles=[f"FOV={f}" for f in FOV_FACTORS],
    suptitle="Experimento A: fov_factor",
)

### Evaluación humana del encuadre

Esta evaluación es deliberadamente manual: estamos calibrando **cómo mostrar** la galaxia antes de
entrenar cualquier modelo. Completar `review_score` inspeccionando las cuadrículas de arriba:

```
-2 = demasiado cerca      -1 = algo cerca      0 = buen encuadre
+1 = algo lejos           +2 = demasiado lejos
```

In [ ]:
review_fov = make_review_table(index_fov, extra_group_field="fov_factor", extra_group_values=FOV_FACTORS)
review_fov

## 7. Experimento B — DR9 vs DR11

**No se ejecuta hasta fijar un `fov_factor` provisional** a partir de la inspección visual del
Experimento A (variable `PROVISIONAL_FOV_FACTOR`, editar arriba en la sección 1 si es necesario).

Mantenemos constantes: mismas galaxias, mismo `fov_factor`, mismo `size_px`, misma composición
(`grz`), y comparamos únicamente el release/capa (`layer`): `ls-dr9` vs `ls-dr11`. **No asumimos que
DR11 es mejor solo por ser más reciente** — lo comprobamos empíricamente para esta muestra MaNGA,
registrando cobertura, artefactos, objetos vecinos y galaxias incompletas.

In [ ]:
jobs_release = []
for release_name, layer in RELEASES.items():
    jobs_release += prepare_jobs(
        sample,
        experiment="release",
        subfolder=release_name.lower(),
        layer=layer,
        bands=BASE_BANDS,
        fov_factor=PROVISIONAL_FOV_FACTOR,
    )

index_release = download_batch(jobs_release)
index_release.head()

In [ ]:
plot_comparison_grid(
    index_release,
    col_field="layer",
    col_values=list(RELEASES.values()),
    col_titles=list(RELEASES.keys()),
    suptitle="Experimento B: DR9 vs DR11",
)

review_release = make_review_table(
    index_release, extra_group_field="layer", extra_group_values=list(RELEASES.values())
)
#review_release

In [ ]:
jobs_release = []
for release_name, layer in RELEASES.items():
    jobs_release += prepare_jobs(
        sample_selected,
        experiment="release",
        subfolder=release_name.lower(),
        layer=layer,
        bands=BASE_BANDS,
        fov_factor=PROVISIONAL_FOV_FACTOR,
    )

index_release = download_batch(jobs_release)
index_release.head()

In [ ]:
plot_comparison_grid(
    index_release,
    col_field="layer",
    col_values=list(RELEASES.values()),
    col_titles=list(RELEASES.keys()),
    suptitle="Experimento B: DR9 vs DR11",
)

review_release = make_review_table(
    index_release, extra_group_field="layer", extra_group_values=list(RELEASES.values())
)

## 8. Experimento C — composición de color

Con `fov_factor` y `layer` ya elegidos provisionalmente (`PROVISIONAL_FOV_FACTOR`,
`PROVISIONAL_LAYER`), comparamos la composición de color: `bands="grz"` explícito vs. **no enviar**
`bands` (render por defecto del Viewer). No asumimos que ambos son equivalentes — por eso el índice
registra explícitamente qué parámetros se enviaron en cada URL (columna `bands_or_color_mode`).

La pregunta: ¿alguna composición hace más visibles brazos, barra, disco o concentración central? Esto
es una inspección de **calidad de entrada**, no una métrica de clasificación.

In [ ]:
jobs_color = []
for mode_name, bands in COLOR_MODES.items():
    jobs_color += prepare_jobs(
        sample,
        experiment="color",
        subfolder=mode_name,
        layer=PROVISIONAL_LAYER,
        bands=bands,
        fov_factor=PROVISIONAL_FOV_FACTOR,
    )

# Documentar explícitamente qué se envió a la URL en cada modo (sección 10).
for job in jobs_color[:2]:
    print(f"{job['bands_or_color_mode']:>8} -> {job['url']}")

index_color = download_batch(jobs_color)
index_color.head()

In [ ]:
plot_comparison_grid(
    index_color,
    col_field="bands_or_color_mode",
    col_values=list(COLOR_MODES.keys()),
    col_titles=list(COLOR_MODES.keys()),
    suptitle="Experimento C: grz explícito vs. default",
)

review_color = make_review_table(
    index_color, extra_group_field="bands_or_color_mode", extra_group_values=list(COLOR_MODES.keys())
)

In [ ]:
jobs_color = []
for mode_name, bands in COLOR_MODES.items():
    jobs_color += prepare_jobs(
        sample_selected,
        experiment="color",
        subfolder=mode_name,
        layer=PROVISIONAL_LAYER,
        bands=bands,
        fov_factor=PROVISIONAL_FOV_FACTOR,
    )

# Documentar explícitamente qué se envió a la URL en cada modo (sección 10).
for job in jobs_color[:2]:
    print(f"{job['bands_or_color_mode']:>8} -> {job['url']}")

index_color = download_batch(jobs_color)
index_color.head()

In [ ]:
plot_comparison_grid(
    index_color,
    col_field="bands_or_color_mode",
    col_values=list(COLOR_MODES.keys()),
    col_titles=list(COLOR_MODES.keys()),
    suptitle="Experimento C: grz explícito vs. default",
)

review_color = make_review_table(
    index_color, extra_group_field="bands_or_color_mode", extra_group_values=list(COLOR_MODES.keys())
)

## 9. Casos extremos: small / medium / large

Reutilizamos las descargas del Experimento A (todas las combinaciones galaxia × `fov_factor`) para
comprobar específicamente si **un único `fov_factor` funciona razonablemente bien en los extremos**:
una galaxia representativa de cada `size_group` (pequeña/mediana/grande), con las cuatro cuadrículas de
FOV juntas. Si, por ejemplo, `FOV=5` funciona en medianas pero falla sistemáticamente en grandes,
documentarlo aquí — no se corrige automáticamente en este playground.

In [ ]:
extreme_names = []
for size_group in ["pequeña", "mediana", "grande"]:
    candidates = sample[sample["size_group"] == size_group]
    if len(candidates):
        extreme_names.append(candidates.iloc[0]["name"])

index_extreme = index_fov[index_fov["name"].isin(extreme_names)]

plot_comparison_grid(
    index_extreme,
    col_field="fov_factor",
    col_values=FOV_FACTORS,
    col_titles=[f"FOV={f}" for f in FOV_FACTORS],
    galaxies_per_grid=len(extreme_names) or 1,
    suptitle="Casos extremos: pequeña / mediana / grande",
)

## 10. Resumen técnico de descargas

Consolidamos el índice de los tres experimentos en un único DataFrame (una fila por
galaxia × configuración, sección 14) y lo guardamos en `playground_index.csv` para poder auditar
exactamente cómo se produjo cada imagen. También calculamos métricas descriptivas simples (brillo medio,
desviación estándar, % de píxeles oscuros/saturados) — son solo apoyo visual, **no** se usan aquí para
declarar automáticamente qué configuración es "mejor".

In [ ]:
playground_index = pd.concat([index_fov, index_release, index_color], ignore_index=True)
PLAYGROUND_DIR.mkdir(parents=True, exist_ok=True)
playground_index.to_csv(PLAYGROUND_INDEX_CSV, index=False)

n_total = len(playground_index)
n_ok = (playground_index["status"] != "error").sum()
n_error = (playground_index["status"] == "error").sum()
n_cached = (playground_index["status"] == "cached").sum()
mb_total = playground_index.loc[playground_index["status"] != "error", "bytes"].sum() / 1024 / 1024
time_total = playground_index["download_time_s"].sum()

print(f"solicitadas : {n_total}")
print(f"correctas   : {n_ok}")
print(f"fallidas    : {n_error}")
print(f"cacheadas   : {n_cached}")
print(f"MB en disco : {mb_total:.2f}")
print(f"tiempo total: {time_total:.1f} s")
print(f"\nÍndice guardado en: {PLAYGROUND_INDEX_CSV}")

In [ ]:
def describe_image_quality(local_path):
    """Métricas descriptivas simples de un JPEG: solo apoyo visual (sección 16)."""
    with Image.open(local_path) as im:
        arr = np.asarray(im.convert("L"), dtype=np.float32)
    return {
        "mean_brightness": float(arr.mean()),
        "std_brightness": float(arr.std()),
        "pct_dark_px": float((arr < 10).mean() * 100),
        "pct_saturated_px": float((arr > 245).mean() * 100),
    }


downloaded_mask = playground_index["status"] != "error"
quality_rows = playground_index.loc[downloaded_mask, "local_path"].apply(describe_image_quality)
quality_df = pd.DataFrame(list(quality_rows), index=playground_index.loc[downloaded_mask].index)
playground_index = playground_index.join(quality_df)

playground_index.to_csv(PLAYGROUND_INDEX_CSV, index=False)
playground_index[
    ["name", "experiment", "fov_factor", "layer", "bands_or_color_mode",
     "mean_brightness", "std_brightness", "pct_dark_px", "pct_saturated_px"]
].head(10)

## 11. Conclusiones provisionales

Completar esta tabla **después** de inspeccionar visualmente las cuadrículas y llenar las tablas de
revisión manual (`review_fov`, `review_release`, `review_color`). Si la evidencia no es clara, escribir
explícitamente "La muestra actual no permite distinguir claramente entre ambas configuraciones" en vez
de forzar una conclusión.

| Decisión | Opciones comparadas | Recomendación provisional | Evidencia |
|---|---|---|---|
| FOV | 4 / 5 / 6 / 8 | 5 a 8 parece un rango razonable | inspección de `plot_comparison_grid` + `review_fov` |
| Release | DR9 / DR11 | No pareciera haber diferencia significativa | inspección de cuadrículas B + `review_release` |
| Color | grz / default | Pareciera que grz es el default | inspección de cuadrículas C + `review_color` |

Este notebook **no entrena modelos, no descarga el dataset completo ni modifica `src/config.py` /
`src/data/legacy_cutouts.py`**. Cuando las tres decisiones queden fijadas, se migran manualmente a esos
archivos siguiendo la regla del proyecto: *lo validado en notebook se migra posteriormente a `src/`.*